# 🧠🩻 Multimodal Deep Learning for Medical Imaging: A Comprehensive Tutorial

---

**Topics covered:**
- What is multimodal learning and why does it matter in medicine?
- Intuition: why a chest X-ray *and* a lab report together beat either alone
- Building a synthetic multimodal dataset (imaging + clinical/tabular data)
- Unimodal baselines (image-only model)
- Fusion strategies: **early**, **late**, and **cross-attention** fusion
- Training and comparing all strategies
- Visualizing what the cross-attention module learns
- Contrastive (CLIP-style) multimodal pretraining — intuition and a toy implementation
- Robustness: what happens when one modality is missing or noisy?
- Summary and practical guidance


## 1. What is Multimodal Learning?

A **modality** is a distinct type/source of data. In medicine, a single patient encounter typically produces *several* modalities at once:

| Modality | Examples |
|---|---|
| **Imaging** | X-ray, CT, MRI, histopathology slide, fundus photo |
| **Tabular / clinical** | Age, vitals, lab values (WBC, CRP, O₂ saturation) |
| **Text** | Radiology report, clinical notes, discharge summary |
| **Genomic / molecular** | Gene expression, mutation panels |
| **Time-series** | ECG, continuous vitals monitoring |

**Multimodal deep learning** builds models that *jointly* consume two or more of these modalities, rather than treating each in isolation. The central bet is that modalities are **complementary**: an X-ray shows *where* something looks abnormal, while labs and vitals describe *how* the patient's physiology is behaving — together they resolve ambiguity that neither resolves alone.

> *A solitary lung opacity on an X-ray could be infection, scarring, or a benign nodule. Add a fever, elevated white-cell count, and falling oxygen saturation, and the diagnosis becomes far less ambiguous.*

This is exactly the synthetic scenario we'll build and model in this notebook.


## 2. Intuition: Why Fuse Modalities?

Think of each modality as a noisy "vote" about the patient's true underlying state $y$ (e.g. disease present / absent).

- The imaging vote $p(y \mid \text{image})$ might be uncertain when an opacity is small or borderline.
- The clinical vote $p(y \mid \text{labs})$ might be uncertain when vitals are only mildly deranged.
- But the **joint** evidence $p(y \mid \text{image}, \text{labs})$ is, under reasonable independence assumptions, *more concentrated* than either marginal:

$$
p(y \mid \text{image}, \text{labs}) \;\propto\; p(\text{image}\mid y)\, p(\text{labs}\mid y)\, p(y)
$$

If the two likelihoods are even weakly informative and not perfectly correlated, their product sharpens the posterior. This is the statistical foundation for *why* fusion tends to outperform single-modality models — and we'll see it empirically later in this notebook.

The **engineering** question is *where* and *how* to combine the modalities inside a neural network. That's what Sections 5–7 are about.


## 3. Environment Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
np.random.seed(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)
print("Torch version:", torch.__version__)

## 4. Building a Synthetic Multimodal Medical Dataset

Real medical imaging datasets (e.g. CheXpert, MIMIC-CXR) require credentialed access and are too large to ship inside a tutorial. Instead, we'll **simulate** a small, controllable multimodal dataset that captures the same *structure* a real one would have:

- **Imaging modality**: synthetic "chest X-ray" — a lung-shaped field with optional simulated opacity (mimicking pneumonia/infiltrate)
- **Clinical modality**: 4 tabular features — age, white blood cell count (WBC), C-reactive protein (CRP), oxygen saturation (O₂ sat)
- **Label** $y$: disease present (1) or absent (0)

Critically, we inject the signal **partially into each modality** and add noise, so that *neither modality alone is fully diagnostic* — exactly the setting where fusion should help.


In [ ]:
def make_synthetic_xray(label, size=64):
    '''Simulate a chest-X-ray-like grayscale image.

    Background = lung field with realistic texture noise.
    If label == 1, a circular 'opacity' (infiltrate) is added at a random location.
    '''
    img = np.random.normal(0.4, 0.05, (size, size)).astype(np.float32)

    # elliptical lung-field mask (brighter than surrounding tissue)
    yy, xx = np.mgrid[0:size, 0:size]
    cx, cy = size / 2, size / 2
    lung_mask = ((xx - cx) ** 2 / (size * 0.35) ** 2 +
                 (yy - cy) ** 2 / (size * 0.45) ** 2) < 1
    img[lung_mask] += 0.15

    if label == 1:
        nx, ny = np.random.randint(size // 4, 3 * size // 4, 2)
        r = np.random.randint(4, 8)
        opacity_mask = (xx - nx) ** 2 + (yy - ny) ** 2 < r ** 2
        img[opacity_mask] += 0.35   # subtle, sometimes small/ambiguous

    img = np.clip(img + np.random.normal(0, 0.03, img.shape), 0, 1)
    return img.astype(np.float32)


def make_synthetic_clinical(label):
    '''Simulate 4 clinical/lab features, mildly shifted by disease status.'''
    age   = np.random.normal(60 if label else 45, 12)
    wbc   = np.random.normal(11.5 if label else 7.5, 1.5)   # x10^9/L
    crp   = np.random.normal(60 if label else 8, 15)         # mg/L
    o2sat = np.random.normal(91 if label else 97, 2)         # %
    return np.array([age, wbc, crp, o2sat], dtype=np.float32)


N = 800
labels = np.random.binomial(1, 0.5, N)
images = np.stack([make_synthetic_xray(l) for l in labels])
clinical = np.stack([make_synthetic_clinical(l) for l in labels])

# standardize clinical features
clinical_mean, clinical_std = clinical.mean(0), clinical.std(0)
clinical_norm = (clinical - clinical_mean) / clinical_std

print("Images:", images.shape, " Clinical:", clinical.shape, " Labels:", labels.shape)
print("Class balance:", labels.mean().round(3))

### 4.1 Visualizing the Two Modalities

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 6))

for col in range(4):
    idx0 = np.where(labels == 0)[0][col]
    idx1 = np.where(labels == 1)[0][col]
    axes[0, col].imshow(images[idx0], cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f"Healthy (id {idx0})", fontsize=9)
    axes[0, col].axis('off')
    axes[1, col].imshow(images[idx1], cmap='gray', vmin=0, vmax=1)
    axes[1, col].set_title(f"Disease (id {idx1})", fontsize=9)
    axes[1, col].axis('off')

fig.suptitle("Synthetic chest-X-ray-like images — opacities are subtle on purpose", y=1.02)
plt.tight_layout(); plt.show()

print("\nClinical feature means by class (age, WBC, CRP, O2 sat):")
print("Healthy:", clinical[labels == 0].mean(0).round(2))
print("Disease:", clinical[labels == 1].mean(0).round(2))

Notice that **neither modality cleanly separates the classes**:
- Some healthy patients have slightly elevated WBC; some diseased patients have small, easy-to-miss opacities.
- This mirrors real clinical ambiguity, and is exactly the regime where combining modalities should pay off.

### 4.2 Train / Validation Split

In [ ]:
idx_tr, idx_va = train_test_split(
    np.arange(N), test_size=0.2, random_state=0, stratify=labels
)

def to_tensors(idx):
    img_t = torch.tensor(images[idx]).unsqueeze(1)        # (B, 1, H, W)
    clin_t = torch.tensor(clinical_norm[idx])              # (B, 4)
    y_t = torch.tensor(labels[idx]).float()                # (B,)
    return img_t, clin_t, y_t

Xi_tr, Xc_tr, y_tr = to_tensors(idx_tr)
Xi_va, Xc_va, y_va = to_tensors(idx_va)

print("Train:", Xi_tr.shape, Xc_tr.shape, y_tr.shape)
print("Val:  ", Xi_va.shape, Xc_va.shape, y_va.shape)

## 5. Modality Encoders

Every fusion architecture starts the same way: each modality gets its **own encoder** that maps raw input into a fixed-size embedding vector $z \in \mathbb{R}^d$.

- **Image encoder**: a small convolutional network (a stand-in for ResNet/EfficientNet/ViT backbones used in practice).
- **Clinical encoder**: a small MLP (a stand-in for embedding tabular/EHR data, or even text embeddings from a clinical language model).

$$
z_{\text{img}} = f_{\theta_{\text{img}}}(\text{image}) \in \mathbb{R}^{d}, \qquad
z_{\text{clin}} = f_{\theta_{\text{clin}}}(\text{clinical}) \in \mathbb{R}^{d}
$$

How we combine $z_{\text{img}}$ and $z_{\text{clin}}$ — and *when* in the network we do it — is the central design decision of multimodal learning.


In [ ]:
class ImageEncoder(nn.Module):
    '''Small CNN backbone -> fixed-size embedding.'''
    def __init__(self, out_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Linear(32, out_dim)

    def forward(self, x):
        h = self.net(x).flatten(1)
        return self.fc(h)


class ClinicalEncoder(nn.Module):
    '''Small MLP for tabular clinical features -> fixed-size embedding.'''
    def __init__(self, in_dim=4, out_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 32), nn.ReLU(), nn.Linear(32, out_dim)
        )

    def forward(self, x):
        return self.net(x)

print(ImageEncoder())
print(ClinicalEncoder())

## 6. Fusion Strategy #1 — Unimodal Baseline (Image Only)

Before fusing anything, we need a baseline that uses **only** the imaging modality. This tells us how much the clinical data is actually contributing later.


In [ ]:
class ImageOnlyModel(nn.Module):
    def __init__(self, dim=32):
        super().__init__()
        self.img_enc = ImageEncoder(dim)
        self.head = nn.Linear(dim, 1)

    def forward(self, img, clin=None):
        return self.head(self.img_enc(img)).squeeze(1)

## 7. Fusion Strategy #2 — Late Fusion (Concatenation)

**Late fusion** runs each encoder independently, then concatenates the two embeddings right before the classification head:

$$
\hat{y} = g_\phi\big([\,z_{\text{img}} \,\Vert\, z_{\text{clin}}\,]\big)
$$

This is the simplest and most common fusion strategy in practice. Each modality keeps its own dedicated backbone, and the network only has to learn how to *weigh* the two embeddings at the very end.

**Pros:** simple, modular, encoders can be pretrained separately.
**Cons:** no interaction between modalities *during* feature extraction — the image encoder never gets to "look at" the clinical values while extracting features, and vice versa.


In [ ]:
class LateFusionModel(nn.Module):
    def __init__(self, dim=32):
        super().__init__()
        self.img_enc = ImageEncoder(dim)
        self.clin_enc = ClinicalEncoder(out_dim=dim)
        self.head = nn.Sequential(
            nn.Linear(dim * 2, 32), nn.ReLU(), nn.Linear(32, 1)
        )

    def forward(self, img, clin):
        zi = self.img_enc(img)
        zc = self.clin_enc(clin)
        fused = torch.cat([zi, zc], dim=1)
        return self.head(fused).squeeze(1)

## 8. Fusion Strategy #3 — Cross-Attention Fusion

**Cross-attention fusion** lets one modality *query* the other, producing an embedding that is conditioned on both. This is the mechanism behind state-of-the-art multimodal architectures (e.g. medical VQA models, report-generation systems, CLIP-style retrieval).

Concretely, we treat the image embedding as a **query** and the clinical embedding as a **key/value**:

$$
Q = z_{\text{img}} W_Q, \quad K = z_{\text{clin}} W_K, \quad V = z_{\text{clin}} W_V
$$

$$
\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right) V
$$

The result is an "attended clinical context" vector, telling the model *how much* to trust the clinical signal given what it sees in the image. We then concatenate this with the raw image embedding before classifying.

To make the attention pattern actually informative, we treat each of the 4 clinical features (age, WBC, CRP, O₂ sat) as its **own token** (rather than collapsing them into one vector first). This way, the image embedding's query can attend *differently* to each individual lab value — exactly as a real cross-attention layer attends over individual image patches or individual words of a report.


In [ ]:
class TokenClinicalEncoder(nn.Module):
    '''Encode each of the 4 clinical features as its own token embedding,
    so cross-attention can weigh individual labs/vitals separately.'''
    def __init__(self, n_feats=4, dim=32):
        super().__init__()
        self.embed = nn.Linear(1, dim)
        self.feat_id = nn.Parameter(torch.randn(n_feats, dim) * 0.1)  # learned positional id per feature

    def forward(self, x):  # x: (B, n_feats)
        tok = self.embed(x.unsqueeze(-1))      # (B, n_feats, dim)
        return tok + self.feat_id.unsqueeze(0)


class CrossAttnFusion(nn.Module):
    def __init__(self, dim=32):
        super().__init__()
        self.img_enc = ImageEncoder(dim)
        self.clin_tok = TokenClinicalEncoder(n_feats=4, dim=dim)
        self.q = nn.Linear(dim, dim)
        self.k = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, dim)
        self.head = nn.Sequential(
            nn.Linear(dim * 2, 32), nn.ReLU(), nn.Linear(32, 1)
        )
        self.dim = dim

    def forward(self, img, clin):
        zi = self.img_enc(img)                  # (B, d)        -- image query source
        tok = self.clin_tok(clin)                # (B, 4, d)     -- one token per clinical feature

        q = self.q(zi).unsqueeze(1)               # (B, 1, d)
        k = self.k(tok)                            # (B, 4, d)
        v = self.v(tok)                            # (B, 4, d)

        scores = (q @ k.transpose(-1, -2)) / np.sqrt(self.dim)   # (B, 1, 4)
        attn = torch.softmax(scores, dim=-1)
        attended = (attn @ v).squeeze(1)            # (B, d)

        fused = torch.cat([zi, attended], dim=1)
        return self.head(fused).squeeze(1), attn.squeeze(1)      # attn: (B, 4)

## 9. Training and Comparing All Three Strategies

In [ ]:
def train_model(model, epochs=30, lr=1e-3, multimodal=True, cross_attn=False):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        if cross_attn:
            out, _ = model(Xi_tr, Xc_tr)
        else:
            out = model(Xi_tr, Xc_tr) if multimodal else model(Xi_tr)
        loss = F.binary_cross_entropy_with_logits(out, y_tr)
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            if cross_attn:
                vout, _ = model(Xi_va, Xc_va)
            else:
                vout = model(Xi_va, Xc_va) if multimodal else model(Xi_va)
            vacc = ((torch.sigmoid(vout) > 0.5).float() == y_va).float().mean().item()
        history.append((loss.item(), vacc))
    return history

torch.manual_seed(1); m_img  = ImageOnlyModel();  h_img  = train_model(m_img,  multimodal=False)
torch.manual_seed(1); m_late = LateFusionModel(); h_late = train_model(m_late)
torch.manual_seed(1); m_attn = CrossAttnFusion();  h_attn = train_model(m_attn, cross_attn=True)

print(f"Final val accuracy — Image only:      {h_img[-1][1]:.3f}")
print(f"Final val accuracy — Late fusion:      {h_late[-1][1]:.3f}")
print(f"Final val accuracy — Cross-attention:  {h_attn[-1][1]:.3f}")

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot([h[1] for h in h_img],  label='Image only',     lw=2)
plt.plot([h[1] for h in h_late], label='Late fusion',    lw=2)
plt.plot([h[1] for h in h_attn], label='Cross-attention', lw=2)
plt.xlabel('Epoch'); plt.ylabel('Validation accuracy')
plt.title('Unimodal vs. Multimodal Fusion: Validation Accuracy')
plt.legend(); plt.grid(alpha=0.3)
plt.ylim(0.4, 1.02)
plt.show()

The gap between the **image-only** curve and the two **fusion** curves is the empirical payoff of multimodality on this dataset: since neither modality alone fully determines $y$, the model that sees both consistently resolves cases that confuse the single-modality model.

> In our synthetic setup, image-only barely beats chance, because we made opacities deliberately subtle. Real disease signal in X-rays is usually much stronger — but the *qualitative* pattern (fusion ≥ best unimodal) holds broadly in published multimodal medical AI literature.


## 10. Inspecting the Cross-Attention Weights

One advantage of attention-based fusion over plain concatenation is **interpretability**: we can directly read off how much weight the model assigns to the clinical modality for each patient.


In [ ]:
m_attn.eval()
with torch.no_grad():
    _, attn_weights = m_attn(Xi_va, Xc_va)   # (B, 4) -- one weight per clinical feature

attn_weights = attn_weights.numpy()
feature_names = ['Age', 'WBC', 'CRP', 'O2 sat']

mean_healthy = attn_weights[y_va.numpy() == 0].mean(0)
mean_disease = attn_weights[y_va.numpy() == 1].mean(0)

x = np.arange(len(feature_names)); width = 0.35
plt.figure(figsize=(7, 4.5))
plt.bar(x - width/2, mean_healthy, width, label='Healthy (y=0)')
plt.bar(x + width/2, mean_disease, width, label='Disease (y=1)')
plt.xticks(x, feature_names)
plt.ylabel('Mean cross-attention weight')
plt.title('Which clinical feature does the model attend to most?')
plt.legend()
plt.show()

print("Mean attention weights (healthy):", mean_healthy.round(3))
print("Mean attention weights (disease):", mean_disease.round(3))

## 11. Robustness: What If a Modality Is Missing or Corrupted?

In real clinical deployment, modalities are frequently **missing** (a lab wasn't drawn yet) or **noisy** (a portable X-ray with motion artifact). A good multimodal system should degrade gracefully rather than catastrophically.

We simulate this by injecting increasing noise into the clinical features at *test time only* (the model was trained on clean data) and see how each fusion strategy holds up.


In [ ]:
noise_levels = [0.0, 0.5, 1.0, 2.0, 4.0]
results = {'Late fusion': [], 'Cross-attention': []}

for sigma in noise_levels:
    Xc_noisy = Xc_va + torch.randn_like(Xc_va) * sigma

    with torch.no_grad():
        out_late = m_late(Xi_va, Xc_noisy)
        acc_late = ((torch.sigmoid(out_late) > 0.5).float() == y_va).float().mean().item()

        out_attn, _ = m_attn(Xi_va, Xc_noisy)
        acc_attn = ((torch.sigmoid(out_attn) > 0.5).float() == y_va).float().mean().item()

    results['Late fusion'].append(acc_late)
    results['Cross-attention'].append(acc_attn)

plt.figure(figsize=(7, 5))
for name, accs in results.items():
    plt.plot(noise_levels, accs, marker='o', label=name, lw=2)
plt.axhline(h_img[-1][1], ls='--', color='gray', label='Image-only (no clinical input)')
plt.xlabel('Clinical-feature noise level (σ)')
plt.ylabel('Validation accuracy')
plt.title('Robustness to Corrupted Clinical Data')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

Both fusion models start above the image-only floor, but as the clinical modality becomes pure noise, performance should converge back toward (and not below) the image-only baseline — a model that has *truly* learned to fuse should never do *worse* than its best single modality, even when the other modality is garbage. If a model instead crashes below the unimodal floor, that's a sign it over-relies on a single modality at training time and hasn't learned a robust fusion (a well-known failure mode called **modality collapse**).


## 12. Beyond Classification: Contrastive Multimodal Pretraining (CLIP-style)

So far we've trained fusion models **end-to-end** for one supervised task. A different and very influential paradigm — popularized by CLIP and adapted to medicine in models like **PubMedCLIP**, **MedCLIP**, and **BioViL** — is to pretrain the two encoders so that **matching** image/clinical (or image/report) pairs are pulled close together in embedding space, and **non-matching** pairs are pushed apart, *without* using disease labels at all.

For a batch of $N$ paired samples, the **contrastive loss** (InfoNCE) is:

$$
\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N} \log \frac{\exp(\text{sim}(z_i^{\text{img}}, z_i^{\text{clin}}) / \tau)}{\sum_{j=1}^{N} \exp(\text{sim}(z_i^{\text{img}}, z_j^{\text{clin}}) / \tau)}
$$

where $\text{sim}(\cdot,\cdot)$ is cosine similarity and $\tau$ is a learned temperature. Intuitively: *each image's embedding should be most similar to its own clinical record's embedding, and dissimilar to every other patient's.*

Let's train a toy version of this on our synthetic data.


In [ ]:
class ProjHead(nn.Module):
    def __init__(self, dim=32, proj_dim=16):
        super().__init__()
        self.fc = nn.Linear(dim, proj_dim)
    def forward(self, x):
        z = self.fc(x)
        return F.normalize(z, dim=-1)

img_enc_c = ImageEncoder(32)
clin_enc_c = ClinicalEncoder(out_dim=32)
proj_img = ProjHead(32, 16)
proj_clin = ProjHead(32, 16)
log_temp = nn.Parameter(torch.tensor(np.log(1/0.07), dtype=torch.float32))

params = list(img_enc_c.parameters()) + list(clin_enc_c.parameters()) + \
         list(proj_img.parameters()) + list(proj_clin.parameters()) + [log_temp]
opt = torch.optim.Adam(params, lr=1e-3)

def contrastive_loss(zi, zc, log_temp):
    temp = log_temp.exp().clamp(max=100)
    logits = (zi @ zc.t()) * temp
    targets = torch.arange(zi.shape[0])
    loss_i = F.cross_entropy(logits, targets)
    loss_c = F.cross_entropy(logits.t(), targets)
    return (loss_i + loss_c) / 2

batch_size = 64
n_steps = 300
clip_losses = []
for step in range(n_steps):
    perm = torch.randperm(Xi_tr.shape[0])[:batch_size]
    img_batch, clin_batch = Xi_tr[perm], Xc_tr[perm]

    zi = proj_img(img_enc_c(img_batch))
    zc = proj_clin(clin_enc_c(clin_batch))
    loss = contrastive_loss(zi, zc, log_temp)

    opt.zero_grad(); loss.backward(); opt.step()
    clip_losses.append(loss.item())

plt.figure(figsize=(6, 4))
plt.plot(clip_losses)
plt.xlabel('Step'); plt.ylabel('Contrastive loss')
plt.title('Toy CLIP-style Pretraining on Image–Clinical Pairs')
plt.show()

### 12.1 Retrieval Sanity Check

If contrastive pretraining worked, each patient's image embedding should be closest to *that same patient's* clinical embedding among a batch of candidates — i.e. we should be able to **retrieve** the correct clinical record given only the image, and vice versa.


In [ ]:
img_enc_c.eval(); clin_enc_c.eval(); proj_img.eval(); proj_clin.eval()
with torch.no_grad():
    n_query = 100
    zi_test = proj_img(img_enc_c(Xi_va[:n_query]))
    zc_test = proj_clin(clin_enc_c(Xc_va[:n_query]))
    sims = zi_test @ zc_test.t()
    top1 = sims.argmax(dim=1)
    retrieval_acc = (top1 == torch.arange(n_query)).float().mean().item()

print(f"Image -> Clinical record retrieval top-1 accuracy: {retrieval_acc:.2%}  "
      f"(chance level = {1/n_query:.2%})")

A retrieval accuracy **above chance** confirms the encoders learned *some* shared structure between modalities — but don't expect it to be dramatic on this synthetic dataset, and that's instructive in itself. Within each class, our synthetic patients are essentially statistical clones of each other (same generating distribution, different noise draws), so there's no patient-*specific* signal for instance-level contrastive learning to lock onto — only class-level signal. Real radiology image–report pairs carry far richer, more idiosyncratic detail (exact lesion location, size, wording), which is *why* contrastive pretraining works so well in practice on real datasets at scale: the more unique, mutually-predictive detail two paired modalities share, the sharper the learned embedding space becomes. This is a useful caveat to keep in mind whenever benchmarking contrastive methods on small or low-diversity datasets.


## 13. Summary

### Fusion Strategy Comparison

| Strategy | When fusion happens | Pros | Cons |
|---|---|---|---|
| **Unimodal** | N/A (single modality) | Simplest, no alignment needed | Ignores complementary information |
| **Early fusion** | Raw input level | Lets the network learn joint low-level features | Requires modalities to be spatially/structurally compatible; brittle to missing modality |
| **Late fusion** | At embedding level (concatenation) | Simple, modular, encoders reusable/pretrainable independently | No cross-modal interaction during feature extraction |
| **Cross-attention fusion** | At embedding level, with learned interaction | Models capture *which* modality to trust per-sample; interpretable attention weights | More parameters, harder to train, needs more data |
| **Contrastive pretraining** | Representation learning stage (before any task head) | Learns a shared embedding space without labels; great for retrieval & transfer | Needs paired data; doesn't directly solve the downstream task by itself |

### Key Takeaways

```
┌────────────────────────────────────────────────────────────────────┐
│              Multimodal Medical Deep Learning Summary              │
├──────────────────────┬───────────────────────────────────────────--┤
│ Core idea            │ Combine complementary modalities to sharpen │
│                       │ predictions beyond what any one modality   │
│                       │ can achieve alone                          │
│ Encoders              │ Each modality gets its own backbone        │
│                       │ (CNN for images, MLP/transformer for       │
│                       │ tabular/text)                              │
│ Fusion point          │ Early (raw input) vs. Late (embeddings)    │
│                       │ vs. Cross-attention (learned interaction)  │
│ Pretraining option    │ Contrastive (CLIP-style) learns a shared   │
│                       │ space without task labels                  │
│ Failure mode to watch │ Modality collapse — model ignores a weaker │
│                       │ modality and gains no robustness from it   │
│ Clinical examples     │ Imaging + labs (pneumonia), imaging +      │
│                       │ genomics (oncology), imaging + reports     │
│                       │ (radiology AI), ECG + vitals (cardiology)  │
└────────────────────────────────────────────────────────────────────┘
```

### Where to Go Next
- Swap the toy CNN/MLP encoders for real backbones (e.g. a pretrained ResNet/ViT for images, a clinical BERT for notes).
- Try **early fusion** by projecting clinical features into spatial maps and concatenating with image channels.
- Explore real multimodal medical datasets: **MIMIC-CXR** (X-ray + reports), **TCGA** (histopathology + genomics), **PTB-XL** (ECG + clinical metadata).
- Read up on **MedCLIP**, **BioViL**, and **Flamingo-CXR** for production-grade multimodal medical architectures.

---
*This notebook used entirely synthetic data for reproducibility and to avoid licensing/PHI concerns — swap in real (de-identified, properly licensed) datasets before drawing clinical conclusions.*
